In [2]:
from pathlib import Path
import rasterio
import numpy as np
import pandas as pd

BASE_DIR = Path(r"C:\Users\Egert\Documents\peatland_detection")

prob_dir = BASE_DIR / "pred_tiles_multiclass_peatmasked_5x5"

out_nodata = -9999.0
threshold = 0.60

rows = []

for tif in sorted(prob_dir.glob("*.tif")):
    with rasterio.open(tif) as src:
        arr = src.read(1).astype("float32")

        nodata = src.nodata
        if nodata is None:
            nodata = out_nodata

        valid = np.isfinite(arr) & (arr != nodata)

        pixel_area_m2 = abs(src.transform.a * src.transform.e)
        pixel_area_ha = pixel_area_m2 / 10000

        valid_pixels = int(valid.sum())
        abandoned_pixels = int((valid & (arr >= threshold)).sum())

        rows.append({
            "tile": tif.name,
            "valid_pixels": valid_pixels,
            "valid_area_ha": valid_pixels * pixel_area_ha,
            "abandoned_pixels": abandoned_pixels,
            "abandoned_area_ha": abandoned_pixels * pixel_area_ha,
            "threshold": threshold
        })

df = pd.DataFrame(rows)

summary = pd.DataFrame([{
    "threshold": threshold,
    "valid_pixels": df["valid_pixels"].sum(),
    "valid_area_ha": df["valid_area_ha"].sum(),
    "abandoned_pixels": df["abandoned_pixels"].sum(),
    "abandoned_area_ha": df["abandoned_area_ha"].sum(),
    "abandoned_share": df["abandoned_area_ha"].sum() / df["valid_area_ha"].sum()
}])

print(df)
print("\nSUMMARY")
print(summary)

out_tile_csv = BASE_DIR / "predicted_abandoned_area_by_tile_5x5_threshold_060.csv"
out_summary_csv = BASE_DIR / "predicted_abandoned_area_summary_5x5_threshold_060.csv"

df.to_csv(out_tile_csv, index=False)
summary.to_csv(out_summary_csv, index=False)

print("\nSaved:", out_tile_csv)
print("Saved:", out_summary_csv)

                tile  valid_pixels  valid_area_ha  abandoned_pixels  \
0  rf_pred_00_00.tif         60245         602.45              5327   
1  rf_pred_00_01.tif         49090         490.90              8495   
2  rf_pred_01_00.tif        501558        5015.58             18395   
3  rf_pred_01_01.tif        598880        5988.80             45730   
4  rf_pred_02_00.tif        376872        3768.72             22580   
5  rf_pred_02_01.tif        309148        3091.48             44091   
6  rf_pred_03_00.tif        256378        2563.78             29486   
7  rf_pred_03_01.tif        305077        3050.77             34792   

   abandoned_area_ha  threshold  
0              53.27        0.6  
1              84.95        0.6  
2             183.95        0.6  
3             457.30        0.6  
4             225.80        0.6  
5             440.91        0.6  
6             294.86        0.6  
7             347.92        0.6  

SUMMARY
   threshold  valid_pixels  valid_area_ha  ab